In [24]:
## run from the terminal
# brew install libomp

## pip install dependencies
# pip install pandas
# pip install yfinance
# pip install dotenv
# pip install transformers
# pip install torch
# pip install joblib
# pip install xgboost
# pip install Flask
# pip install flask_bootstrap
# pip install matplotlib
# pip install ollama
# ollama pull llama3.2


# import libraries (pip install dependencies)
import os
import requests
import pandas as pd
from pandas.tseries.offsets import BDay
import numpy as np
import yfinance as yf
from datetime import datetime
from dotenv import load_dotenv
from transformers import pipeline
import torch
import joblib
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import base64
import json

# new day catalyst import

load_dotenv()
api_key = os.getenv("api_key")

In [25]:
finbert = pipeline(task = "text-classification",
                model = "ProsusAI/finbert",
                tokenizer = "ProsusAI/finbert",
                )

# upload models
def eval_rank_ic(*args, **kwargs):
    pass
models = joblib.load('xgb_models_1_to_20.joblib')

# feartures
features = ['polarity',
            # 'drug_count',
            'Total Revenue', 
            'Operating Revenue', 
            'Cash And Cash Equivalents', 
            'Capital Expenditure', 
            'Stock_Trend_30d', 
            'Stock_Trend_60d',
            # 'NBI_Trend_30d', 
            'NBI_Trend_60d'
            # 'isDelisted'
            ]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 53773.13it/s]


In [26]:
url = f"https://api.biopharmcatalyst.com/api/user/v1/historical-catalysts"

params = {
    "key": api_key
}

response = requests.get(url, params = params)
res = response.json()

df = pd.DataFrame(res)

In [27]:
### extract company features
def getFinancialInfo(tickers):
    financial_data = []
    for ticker in tickers:
        company = yf.Ticker(ticker)
        row = {"company_ticker": ticker}
        for statement in [company.financials, company.balance_sheet, company.cashflow]:
            for col in statement.columns:
                if col.year == 2024:
                    row.update(statement[col])
        financial_data.append(row)
    financial_data = pd.DataFrame(financial_data).reindex(columns = 
            ['company_ticker', 
            'Total Revenue',  
            'Operating Revenue',
            'Cash And Cash Equivalents',
            'Total Debt',
            'Capital Expenditure'])
    return financial_data

In [28]:
all_data = pd.DataFrame()

for days_lookback in range(20,60):

    selected_trading_date = (pd.offsets.BDay().rollback(pd.Timestamp.now().normalize() - pd.offsets.BDay(days_lookback))).strftime('%Y-%m-%d')

    print(f"Selected trading day: {selected_trading_date}")

    new_data = df[df["date"] == selected_trading_date].copy()

    if new_data.empty:
        print("No data.")
        continue

    new_data = new_data.drop_duplicates(
        subset=['company_ticker', 'date'], keep = 'last').reset_index(drop=True)

    print(f"Found {len(new_data)} catalysts for {selected_trading_date}")

    # calculate polarity
    if not new_data.empty:
        output = finbert(new_data['catalyst'].astype(str).to_list(), batch_size = 32, truncation = True, top_k = None)

        new_data['polarity'] = [{s['label']:s['score'] for s in opt}['positive']-
        {s['label']:s['score'] for s in opt}['negative']
        for opt in output]
    else:
        new_data['polarity'] = []

    company_features = getFinancialInfo(new_data['company_ticker']).dropna()

    new_data['date'] = pd.to_datetime(new_data['date'])

    # drug_count = new_data[new_data['date'].dt.year.isin([2024])].groupby('company_ticker')['drug_name'].nunique()

    # map back to the df company table

    # company_features['drug_count'] = company_features['company_ticker'].map(drug_count).fillna(0).astype(int)

    # merge with historical 2025 company features

    new_data['company_ticker'] = new_data['company_ticker'].astype(str)
    company_features['company_ticker'] = company_features['company_ticker'].astype(str)

    new_data = pd.merge(
        new_data,
        company_features,
        on='company_ticker',
        how = 'left',
    )

    # updated market data
    market_data = []    
    for ticker in new_data['company_ticker'].unique():
        hist = yf.Ticker(ticker).history(start = pd.to_datetime(selected_trading_date) - pd.DateOffset(300), end = pd.to_datetime(selected_trading_date) + pd.offsets.BDay(22)).reset_index()
        hist['Ticker'] = ticker
        hist['Date'] = pd.to_datetime(hist['Date'], utc = True).dt.tz_localize(None).dt.normalize()
        keptColumns = ['Date', 'Ticker', 'Open', 'Close', 'High', 'Low', 'Volume']
        hist = hist.reindex(columns = keptColumns)
        market_data.append(hist)
    if market_data:
        market_data_df = pd.concat(market_data, ignore_index=True)
    else:
        market_data_df = pd.DataFrame(columns = keptColumns)

    nbi = yf.Ticker('^NBI').history(start = pd.to_datetime(selected_trading_date) - pd.DateOffset(300), end = pd.to_datetime(selected_trading_date)+ pd.offsets.BDay(22)).reset_index()
    nbi = nbi[['Date', 'Close']].rename(columns= {'Close':'NBI'})
    nbi['Date'] = pd.to_datetime(nbi['Date'], utc = True).dt.tz_localize(None).dt.normalize()
    market_data_df = pd.merge(market_data_df, nbi, on= 'Date', how='left')
    

    # add 30d/60d trends
    def trend(market_data_df, ticker, eventDate):
        df = (market_data_df[market_data_df['Ticker'] == ticker].sort_values('Date').reset_index(drop=True))
 
        event_date = pd.to_datetime(eventDate).normalize()
        t0_sb = df[df['Date'] <= event_date]

        if t0_sb.empty:
            return {
                'Stock_Trend_30d': np.nan,
                'Stock_Trend_60d': np.nan,
                # 'NBI_Trend_30d': np.nan,
                'NBI_Trend_60d': np.nan
                # 'isDelisted': 1
        }
        
        t0_idx = t0_sb.index[-1]
        
        # extract stock values at specified timepoints
        n=len(df)
        closing_price = df.iloc[t0_idx+1]['Close']
        s0 = df.iloc[t0_idx]['Close']
        s30 = df.iloc[t0_idx-21]['Close'] if t0_idx>=21 else np.nan
        s60 = df.iloc[t0_idx-42]['Close'] if t0_idx>=41 else np.nan
        n0 = df.iloc[t0_idx].get('NBI', np.nan)
        n30 = df.iloc[t0_idx-21].get('NBI', np.nan) if t0_idx>=21 else np.nan
        n60 = df.iloc[t0_idx-42].get('NBI', np.nan) if t0_idx>=41 else np.nan

        future_prices = df.loc[t0_idx+1 : t0_idx+21, 'Close'].values
        p = {f"p{i}": future_prices[i-1] if i - 1 < len(future_prices) else np.nan for i in range(1,21)}

        # calculate trends
        return {
            'Closing Price': closing_price,
            'Stock_Trend_30d': (s0-s30) / s30 if pd.notna(s30) and s30!=0 else np.nan,
            'Stock_Trend_60d': (s0-s60) / s60 if pd.notna(s60) and s60!=0 else np.nan,
            # 'NBI_Trend_30d': (n0-n30) / n30 if pd.notna(n30) and n30!=0 else np.nan,
            'NBI_Trend_60d': (n0-n60) / n60 if pd.notna(n60) and n60!=0 else np.nan,
            # 'isDelisted': pd.isna(s30) or pd.isna(s60),
            **p
        }

    trends_res = [trend(market_data_df, ticker, date)
                            for ticker, date in zip(new_data['company_ticker'], 
                                                    new_data['date'])]
    trends_df = pd.DataFrame([res if isinstance(res, dict) else {} for res in trends_res], index = new_data.index)

    new_data = pd.concat([new_data, trends_df], axis = 1)

    for c in features:
        if new_data[c].dtype == 'object':
            new_data[c] = new_data[c].astype('category')

    # new data
    X_new = new_data[features].copy()
    # X_new['isDelisted'] = X_new['isDelisted'].astype(int)

    # run regressor predictions
    preds_df = pd.DataFrame({f"NCAR{i}": model.predict(X_new) for i, model in enumerate(models.values(), start=1)})
    new_data = pd.concat([new_data, preds_df], axis =1)

    # quintile 1 is the highest
    new_data["NCAR20_quintiles"] = np.nan
    if (len(new_data) >=5):
        new_data["NCAR20_quintiles"] = pd.qcut(new_data["NCAR20"].rank(method = "first"), q=5, labels = [5,4,3,2,1])
    else: 
        print("Fewer than 5 catalysts for the last trading day")


    all_data = pd.concat([all_data, new_data], ignore_index=True)

all_data.shape   

Selected trading day: 2026-08-28
Found 8 catalysts for 2026-08-28
Selected trading day: 2026-08-27
Found 14 catalysts for 2026-08-27
Selected trading day: 2026-08-26
Found 6 catalysts for 2026-08-26
Selected trading day: 2026-08-25
Found 5 catalysts for 2026-08-25
Selected trading day: 2026-08-24
Found 9 catalysts for 2026-08-24
Selected trading day: 2026-08-21
Found 1 catalysts for 2026-08-21
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-20
Found 3 catalysts for 2026-08-20
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-19
Found 9 catalysts for 2026-08-19
Selected trading day: 2026-08-18
Found 7 catalysts for 2026-08-18
Selected trading day: 2026-08-17
Found 16 catalysts for 2026-08-17


$PSTV: possibly delisted; no price data found  (1d 2025-10-21 00:00:00 -> 2026-09-16 00:00:00)
$GLMD: possibly delisted; no price data found  (1d 2025-10-21 00:00:00 -> 2026-09-16 00:00:00)


Selected trading day: 2026-08-14
Found 2 catalysts for 2026-08-14
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-13
Found 5 catalysts for 2026-08-13
Selected trading day: 2026-08-12
Found 3 catalysts for 2026-08-12
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-11
Found 6 catalysts for 2026-08-11
Selected trading day: 2026-08-10
Found 10 catalysts for 2026-08-10
Selected trading day: 2026-08-07
Found 4 catalysts for 2026-08-07
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-06
Found 16 catalysts for 2026-08-06
Selected trading day: 2026-08-05
Found 6 catalysts for 2026-08-05
Selected trading day: 2026-08-04
Found 4 catalysts for 2026-08-04
Fewer than 5 catalysts for the last trading day
Selected trading day: 2026-08-03
Found 10 catalysts for 2026-08-03
Selected trading day: 2026-07-31
Found 5 catalysts for 2026-07-31
Selected trading day: 2026-07-30
Found 14 catalysts for 2026-07-30
Selected tra

$NUVL: possibly delisted; no price data found  (1d 2025-09-25 00:00:00 -> 2026-08-21 00:00:00)


Selected trading day: 2026-07-21
Found 13 catalysts for 2026-07-21


$CYBN: possibly delisted; no timezone found


Selected trading day: 2026-07-20
Found 15 catalysts for 2026-07-20


$CLSD: possibly delisted; no timezone found


Selected trading day: 2026-07-17
Found 8 catalysts for 2026-07-17
Selected trading day: 2026-07-16
Found 7 catalysts for 2026-07-16
Selected trading day: 2026-07-15
Found 18 catalysts for 2026-07-15
Selected trading day: 2026-07-14
Found 8 catalysts for 2026-07-14
Selected trading day: 2026-07-13
Found 20 catalysts for 2026-07-13
Selected trading day: 2026-07-10
Found 7 catalysts for 2026-07-10
Selected trading day: 2026-07-09
Found 11 catalysts for 2026-07-09
Selected trading day: 2026-07-08
Found 9 catalysts for 2026-07-08
Selected trading day: 2026-07-07
Found 11 catalysts for 2026-07-07
Selected trading day: 2026-07-06
Found 10 catalysts for 2026-07-06


(375, 67)

In [29]:
def backtest(data, stopLoss, takeProfit):
    total_value = 10000
    returns = []
    holdingDays = []

    for row in data.itertuples():
        acquisition_price = getattr(row, 'p2', None)
        if acquisition_price is None or pd.isna(acquisition_price):
            continue

        profit_loss = None
        daysHeld = 19

        for i in range(3,21):
            price = getattr(row, f"p{i}", None)
            if price is None or pd.isna(price):
                continue
            if price <= acquisition_price*(1-stopLoss):
                profit_loss = -stopLoss
                daysHeld = i-1
                break
            elif price >= acquisition_price*(1+takeProfit):
                profit_loss = takeProfit
                daysHeld = i-1
                break
        if profit_loss is None:
            price_Day20 = getattr(row, "p20", None)
            if price_Day20 is not None and not pd.isna(price_Day20):
                profit_loss = (price_Day20 - acquisition_price) / acquisition_price
            else:
                profit_loss = 0 
             
        returns.append(profit_loss)
        holdingDays.append(daysHeld)
        total_value =   total_value + 10000 * profit_loss

    if (len(returns)==0):
        return {
        "Total Return": 0,
        "Sharpe Ratio": -np.inf,
        "Win Rate": 0,
        "Average Holding Days": 0,
        "Profit Factor": 0,
        "Total Trades": 0,
        }

    returns = np.array(returns)
    holdingDays = np.array(holdingDays)

    avg_returns = np.mean(returns)
    std_returns = np.std(returns)
    sharpe = (avg_returns/std_returns) if std_returns > 0 else -np.inf
    # totalPercReturn = (total_value - 10000)/ 10000
    totalPercReturn = np.sum(returns)

    wins = returns > 0
    losses = returns < 0
    winRate = np.mean(wins)
    avgHoldingDays = np.mean(holdingDays)

    totalGains = np.sum(returns[wins])
    totalLosses = - np.sum(returns[losses])
    profitFactor = (totalGains / totalLosses) if totalLosses > 0 else np.inf


    results = {
        "Total Return": totalPercReturn,
        "Sharpe Ratio": sharpe,
        "Win Rate": winRate,
        "Average Holding Days": avgHoldingDays,
        "Profit Factor": profitFactor,
        "Total Trades": len(returns),
    }

    return results

In [30]:
all_data.columns

Index(['nct_number', 'company_name', 'company_ticker', 'conference_acronym',
       'conference_end_date', 'conference_link', 'conference_name',
       'conference_start_date', 'date', 'drug_name', 'indication', 'label',
       'press_link', 'catalyst', 'stage', 'statuses', 'polarity',
       'Total Revenue', 'Operating Revenue', 'Cash And Cash Equivalents',
       'Total Debt', 'Capital Expenditure', 'Closing Price', 'Stock_Trend_30d',
       'Stock_Trend_60d', 'NBI_Trend_60d', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6',
       'p7', 'p8', 'p9', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16',
       'p17', 'p18', 'p19', 'p20', 'NCAR1', 'NCAR2', 'NCAR3', 'NCAR4', 'NCAR5',
       'NCAR6', 'NCAR7', 'NCAR8', 'NCAR9', 'NCAR10', 'NCAR11', 'NCAR12',
       'NCAR13', 'NCAR14', 'NCAR15', 'NCAR16', 'NCAR17', 'NCAR18', 'NCAR19',
       'NCAR20', 'NCAR20_quintiles'],
      dtype='str')

In [31]:
# returns on the test data set using predicted NCAR 20 quintiles

optimal_SL = 0.28
optimal_TP = 0.4

all_data = all_data[all_data['NCAR20_quintiles'].astype(float) == 1]
print(backtest(all_data, optimal_SL, optimal_TP))

{'Total Return': np.float64(2.6314104561235525), 'Sharpe Ratio': np.float64(0.23674287067541722), 'Win Rate': np.float64(0.527027027027027), 'Average Holding Days': np.float64(17.77027027027027), 'Profit Factor': np.float64(2.0121402511547286), 'Total Trades': 74}
